# 01 · Data exploration

A look at the event data behind the system: UCDP GED 25.1, filtered to events with Ukraine as a conflict party, 2022–2024. The findings here shaped three design choices:

1. **The source-affiliation table** (`config/source_affiliations.yaml`) — which outlets need a category.
2. **The document template** (`preprocessing.py`) — which fields to embed and how long documents get.
3. **The router's filters** — which places and years the data actually contains.

**Needs:** only the repository (the event table ships with it). Run the cells top to bottom.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

from rag_ucdp.config import DEFAULT_CONFIG_PATH, load_config, resolve_path
from rag_ucdp.data_loading import load_processed

cfg = load_config(DEFAULT_CONFIG_PATH)
df = load_processed(resolve_path(cfg.data.processed_file))
print(f"{len(df):,} events, {df['date_start'].min():%Y-%m-%d} to {df['date_start'].max():%Y-%m-%d}")
print(f"{len(df.columns)} columns")

**Where the table comes from.** `rag-ucdp data` downloads the UCDP release, keeps events where Ukraine is one of the two sides within the configured dates, and adds two columns: a readable violence type and the list of reporting outlets. Scoping by *party* rather than by country also captures cross-border events in Russia and Poland, as shown below.

## 1. Who reports the events?

Each event lists its sources in `source_office` (`;`-separated), already split into `source_office_list`.

In [ ]:
source_counts = Counter(o for outlets in df["source_office_list"] for o in outlets)
total = sum(source_counts.values())
print(f"{total:,} source mentions from {len(source_counts):,} distinct outlets\n")

print("Top 15 outlets:")
for i, (name, n) in enumerate(source_counts.most_common(15), 1):
    print(f"  {i:>2}. {name[:50]:<50} {n:>6,}")

In [ ]:
for k in (10, 30, 100, 300):
    covered = sum(n for _, n in source_counts.most_common(k))
    print(f"Top {k:>3} outlets cover {100 * covered / total:5.1f}% of mentions")

per_event = df["source_office_list"].apply(len)
print(f"\nSources per event: median {per_event.median():.0f}, mean {per_event.mean():.2f}")

**What this means for the design**

- **A short hand-curated table is enough.** The top 30 outlets cover about 94% of all mentions, so the affiliation table classifies the frequent outlets and puts the long tail of roughly 900 rare ones into *other / unknown*.
- **Bias has to be measured across events, not within them.** The typical event has a single source, so the source-bias panel looks at the mix across all retrieved events.
- **Outlet names vary in spelling** (e.g. `UA Losses` / `UALosses`), so lookups are case- and whitespace-insensitive.
- **The real mix differs from intuition.** It includes casualty trackers and OSINT projects alongside state media, and e.g. Mediazona is a Russian *independent* outlet, not a state one. The categories in the table follow the data.

## 2. How much text does each event carry?

Retrieval embeds one short document per event. The question is which text fields are filled and whether documents fit BGE-large's 512-token input.

In [ ]:
text_fields = ["source_headline", "where_description", "adm_1", "adm_2", "source_article"]
for col in text_fields:
    filled = (df[col].notna() & (df[col].astype(str).str.strip() != "")).mean()
    print(f"  {col:<18} filled in {100 * filled:5.1f}% of events")

In [ ]:
from rag_ucdp.preprocessing import build_document_text

docs = df.apply(build_document_text, axis=1)
chars = docs.str.len()
print(f"Document length (characters): median {int(chars.median())}, "
      f"p99 {int(chars.quantile(0.99))}, max {chars.max()}")
print(f"Rough tokens (≈ chars / 4):   median {int(chars.median()) // 4}, "
      f"max {chars.max() // 4}\n")
print("Example document:\n")
print(docs.iloc[0])

**What this means for the design**

- **The headline is the only rich free text**, and it is what makes semantic search worthwhile; the other fields are structured and better served by filters.
- **Documents are short** (about 110 tokens at the median), well inside BGE-large's limit. Very long headline lists are truncated in `build_document_text`, so even the longest document fits.
- **Empty fields are skipped** rather than written as placeholders, e.g. the specific location, which is missing for about 3% of events.

## 3. Where and when?

In [ ]:
print("Events by country:")
print(df["country"].value_counts().to_string())
print("\nTop 10 oblasts:")
print(df["adm_1"].value_counts().head(10).to_string())

In [ ]:
monthly = pd.to_datetime(df["date_start"]).dt.to_period("M").value_counts().sort_index()

fig, ax = plt.subplots(figsize=(11, 3.5))
monthly.plot(ax=ax, marker="o", markersize=3, linewidth=1.5, color="#c0392b")
ax.set_title("UCDP events with Ukraine as a party, per month")
ax.set_xlabel("")
ax.set_ylabel("Events")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Events per month: median {int(monthly.median())}, "
      f"min {monthly.min()} ({monthly.idxmin()}), max {monthly.max()} ({monthly.idxmax()})")

**What this means for the design**

- **Cross-border events are captured:** 777 events lie in Russia (mostly Kursk and Belgorod oblasts) and one in Poland (the Przewodów missile incident, November 2022). The router therefore knows Russian border oblasts too.
- **Donetsk oblast dominates** with about half of all events. An unfiltered question such as *"shelling of residential areas"* will mostly return Donetsk events, which is why the router extracts place filters and applies them *before* ranking.
- **The data is dense** (several hundred events in a typical month), so even narrow year or place filters leave plenty to retrieve from.

**Next:** [`02_retrieval_and_answering.ipynb`](02_retrieval_and_answering.ipynb) walks through how a question is routed and answered.